In [4]:
import os, sys
from pathlib import Path

project_root = Path.cwd().parent   
sys.path.insert(0, str(project_root))

from transformers import AutoTokenizer  # type:ignore
from core.config import MODEL_DIR

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)

# 用一个最简单的两轮对话来渲染
sample = [
    {"role": "user", "content": "你好"},
    {"role": "assistant", "content": "你好，有什么可以帮你的？"},
]

rendered = tokenizer.apply_chat_template(sample, tokenize=False, add_generation_prompt=False)

print("=" * 60)
print("【渲染出的原始文本，repr形式（能看到不可见字符）】")
print(repr(rendered))
print("=" * 60)

# 你手写的两个marker
hand_start = "<bos>assistant\n"
hand_end = "<eos>\n"

print("手写 start marker 是否出现在渲染文本里:", hand_start in rendered)
print("手写 end marker   是否出现在渲染文本里:", hand_end in rendered)

【渲染出的原始文本，repr形式（能看到不可见字符）】
'<|im_start|>user\n你好<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n你好，有什么可以帮你的？<|im_end|>\n'
手写 start marker 是否出现在渲染文本里: False
手写 end marker   是否出现在渲染文本里: False


/home/user/anaconda3/envs/nlpV2_wen/compiler_compat/ld: cannot find -laio: 没有那个文件或目录
collect2: error: ld returned 1 exit status
/home/user/anaconda3/envs/nlpV2_wen/compiler_compat/ld: cannot find -laio: 没有那个文件或目录
collect2: error: ld returned 1 exit status


In [5]:
import json
has_r, no_r, total = 0, 0, 0
file = '/home/user/data/2025/wen/train_llm/data/raw/sft_t2t_mini.jsonl'
with open(file, encoding="utf-8") as f:
    for line in f:
        sample = json.loads(line)
        for msg in sample["conversations"]:
            if msg["role"] == "assistant":
                total += 1
                if msg.get("reasoning_content"):
                    has_r += 1
                else:
                    no_r += 1
print(f"总assistant轮次:{total}, 有reasoning:{has_r}({has_r/total:.1%}), 无:{no_r}")

总assistant轮次:1249691, 有reasoning:308390(24.7%), 无:941301


In [6]:
sample_with_reasoning = [
    {"role": "user", "content": "训练数据来源是什么？"},
    {
        "role": "assistant",
        "content": "我的训练数据涵盖多领域，确保覆盖广泛，但具体细节不公开。",
        "reasoning_content": "好的，用户问训练数据来源...",  # 直接把原始字段传进去，不要拼接
    },
]
rendered = tokenizer.apply_chat_template(sample_with_reasoning, tokenize=False, add_generation_prompt=False)
print(repr(rendered))

'<|im_start|>user\n训练数据来源是什么？<|im_end|>\n<|im_start|>assistant\n<think>\n好的，用户问训练数据来源...\n</think>\n\n我的训练数据涵盖多领域，确保覆盖广泛，但具体细节不公开。<|im_end|>\n'


In [7]:
print("bos_token:", repr(tokenizer.bos_token))
print("eos_token:", repr(tokenizer.eos_token))
print("pad_token:", repr(tokenizer.pad_token))
print("pad_token_id == eos_token_id ?", tokenizer.pad_token_id == tokenizer.eos_token_id)

bos_token: '<|im_start|>'
eos_token: '<|im_end|>'
pad_token: '<|endoftext|>'
pad_token_id == eos_token_id ? False


In [8]:
print("tokenizer 实际词表大小:", tokenizer.vocab_size)
print("tokenizer len():", len(tokenizer))
print("配置的 vocab_size:", 6400)

tokenizer 实际词表大小: 6400
tokenizer len(): 6400
配置的 vocab_size: 6400
